# Evaluate with different metrics

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from collections import defaultdict
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import pickle
import inspect
import sys
sys.path.append("..")
sys.path.append("../prediction_src")

from fit_prediction_model import *
from evaluate_prediction_model import *
from params import *

from calibration import classification_metrics

### Load Models

In [3]:
# Alle DAG-Prädiktoren OHNE Durchfall OHNE Dyspareunie:
with open("../models/DAG_model_dyspareunie.pkl","rb") as f:
    dyspareunie_model_dict = pickle.load(f)

In [4]:
# Alle DAG-Prädiktoren OHNE Durchfall OHNE Urinieren:
with open("../models/DAG_model_urinieren.pkl","rb") as f:
    urinieren_model_dict = pickle.load(f)

In [5]:
# Nur Menstruationsschmerz als Prädiktor:
with open("../models/menstruationsschmerz_predictor.pkl","rb") as f:
    menstruationsschmerz_dict = pickle.load(f)

In [6]:
# Alle DAG-Prädiktoren OHNE Durchfall:
with open("../models/DAG_model_durchfall.pkl","rb") as f:
    durchfall_model_dict= pickle.load(f)

In [7]:
# Alle DAG-Prädiktoren OHNE Durchfall MIT youden cutoff:
with open("../models/DAG_model_youden_final.pkl","rb") as f:
    youden_model_dict = pickle.load(f)

In [8]:
# Alle DAG-Prädiktoren mit zusätzlichen NVF-Variablen:
with open("../models/DAG_model_nvf.pkl","rb") as f:
    nvf_model_dict = pickle.load(f)

In [9]:
# Alle DAG-Prädiktoren:
with open("../models/DAG_model_thresh_new.pkl","rb") as f:
    model_dict = pickle.load(f)

### Give each model a readable name

In [10]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model_name"]=get_df_name(durchfall_model_dict)

In [11]:
model_dict[list(model_dict.keys())[0]]["model_name"]=get_df_name(model_dict)

In [12]:
youden_model_dict[list(youden_model_dict.keys())[0]]["model_name"]=get_df_name(youden_model_dict)

In [13]:
nvf_model_dict[list(nvf_model_dict.keys())[0]]["model_name"]=get_df_name(nvf_model_dict)

In [14]:
urinieren_model_dict[list(urinieren_model_dict.keys())[0]]["model_name"]=get_df_name(urinieren_model_dict)

In [15]:
dyspareunie_model_dict[list(dyspareunie_model_dict.keys())[0]]["model_name"]=get_df_name(dyspareunie_model_dict)

In [16]:
menstruationsschmerz_dict[list(menstruationsschmerz_dict.keys())[0]]["model_name"]=get_df_name(menstruationsschmerz_dict)

### Calculate evaluation metrics

In [17]:
mens_pred_metrics_of_interest = calc_eval_metrics(menstruationsschmerz_dict,MET_LIST)
durchfall_metrics_of_interest = calc_eval_metrics(durchfall_model_dict,MET_LIST)
full_metrics_of_interest = calc_eval_metrics(model_dict,MET_LIST)
nvf_metrics_of_interest = calc_eval_metrics(nvf_model_dict,MET_LIST)
dyspareunie_metrics_of_interest = calc_eval_metrics(dyspareunie_model_dict,MET_LIST)
urinieren_metrics_of_interest = calc_eval_metrics(urinieren_model_dict,MET_LIST)
youden_metrics_of_interest = calc_eval_metrics(youden_model_dict,MET_LIST)


### Name the results

In [18]:
mens_pred_metrics_of_interest.reset_index(inplace=True)
mens_pred_metrics_of_interest["Model"]="Menstruationsschmerz einziger Prädiktor"

In [19]:
nvf_metrics_of_interest.reset_index(inplace=True)
nvf_metrics_of_interest["Model"]="zusätzliche NVF-Prädiktoren"

In [20]:
full_metrics_of_interest.reset_index(inplace=True)
full_metrics_of_interest["Model"]="Alle DAG-Prädiktoren"

In [21]:
durchfall_metrics_of_interest.reset_index(inplace=True)
durchfall_metrics_of_interest["Model"]="DAG-Prädiktoren ohne Durchfall"

In [22]:
urinieren_metrics_of_interest.reset_index(inplace=True)
urinieren_metrics_of_interest["Model"]="DAG-Prädiktoren ohne Durchfall ohne Urinieren"

In [23]:
dyspareunie_metrics_of_interest.reset_index(inplace=True)
dyspareunie_metrics_of_interest["Model"]="DAG-Prädiktoren ohne Durchfall ohne Dyspareunie"

In [24]:
youden_metrics_of_interest.reset_index(inplace=True)
youden_metrics_of_interest["Model"]="DAG-Prädiktoren ohne Durchfall mit youden cutoff"

### Show evaluation results

In [28]:
all_metrics_of_interest = pd.concat([full_metrics_of_interest,mens_pred_metrics_of_interest,nvf_metrics_of_interest,
                                     durchfall_metrics_of_interest, youden_metrics_of_interest,urinieren_metrics_of_interest,
                                    dyspareunie_metrics_of_interest])
all_metrics_of_interest

,Model,AUC,Brier Score Loss,Calibration Intercept,Calibration Slope
0,Alle DAG-Prädiktoren,0.656,0.02,0.269,1.076
0,Menstruationsschmerz einziger Prädiktor,0.595,0.02,0.508,1.136
0,zusätzliche NVF-Prädiktoren,0.674,0.02,0.120,1.035
0,DAG-Prädiktoren ohne Durchfall,0.656,0.02,0.259,1.073
0,DAG-Prädiktoren ohne Durchfall mit youden cutoff,0.656,0.02,0.259,1.073
0,DAG-Prädiktoren ohne Durchfall ohne Urinieren,0.654,0.02,0.355,1.099
0,DAG-Prädiktoren ohne Durchfall ohne Dyspareunie,0.656,0.02,0.247,1.070


# Vorhersagewerte für Testset

In [28]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model"].threshold

0.5

In [29]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["prob_1_test"].mean()

np.float64(0.021100423371885432)

In [30]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["pred_test"].mean()

np.float64(0.0)

In [31]:
X_test_true = durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["X_test_true"]

In [32]:
Mens_test_true = X_test_true[X_test_true["Menstruationsschmerzen"]==1].copy()

In [33]:
X_test_true.shape, Mens_test_true.shape

((19974, 7), (2432, 7))

In [34]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model"].predict(X_test_true).mean().round(3)

np.float64(0.0)

In [35]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model"].predict(Mens_test_true).mean()

np.float64(0.0)

In [36]:
pd.Series(durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model"].predict_proba(X_test_true)[:,1]).mean().round(3)

np.float64(0.021)

In [37]:
pd.Series(durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model"].predict_proba(Mens_test_true)[:,1]).mean().round(3)

np.float64(0.048)

In [39]:
durchfall_model_dict[list(durchfall_model_dict.keys())[0]]["model"].threshold

0.5